# Nepali Unigram Harvester & Frequency Counter

Extract, clean, and harvest unigram word frequency distributions from large Nepali text corpora (e.g., Hugging Face datasets or local text files).

### Pipeline:
1. **Devanagari Normalization & Tokenization**: Strips punctuation (Purnabiram `।`, symbols, digits) and extracts Devanagari word sequences (`[\u0900-\u097F]+`).
2. **Harvesting & Counting**: Aggregates word frequencies with `collections.Counter` and progress tracking (`tqdm`).
3. **Export**: Saves frequencies to TSV (`unigram\tcount`) sorted in descending order for vocabulary creation or NLP workflows.

## 1. Setup & Imports

In [ ]:
# Install required packages if not already present in your environment:
# !pip install datasets tqdm

import re
from collections import Counter
from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset

## 2. Text Preprocessing & Devanagari Tokenization

In [ ]:
# Compile regex patterns for Devanagari word extraction
PURNABIRAM_PATTERN = re.compile(r"\u0964")
NEPALI_WORD_PATTERN = re.compile(r"[\u0900-\u097F]+")

def tokenize_nepali_unigrams(text: str) -> list[str]:
    """
    Cleans and tokenizes Nepali text into unigram tokens.
    Removes Purnabiram and extracts contiguous Devanagari character sequences.
    """
    if not isinstance(text, str) or not text.strip():
        return []
    
    # Remove Devnagari Purnabiram (।)
    cleaned = PURNABIRAM_PATTERN.sub(" ", text)
    
    # Extract all Devanagari word tokens
    words = NEPALI_WORD_PATTERN.findall(cleaned)
    return words

# Quick example check
sample_text = "यो एउटा उदाहरण पाठ हो। यसमा केही नेपाली शब्दहरू छन्।"
print("Sample input:", sample_text)
print("Tokenized unigrams:", tokenize_nepali_unigrams(sample_text))

## 3. Load Nepali Corpus

Using `IRIIS-RESEARCH/Nepali-Text-Corpus` from Hugging Face Hub (or configure your own dataset split).

In [ ]:
DATASET_NAME = "IRIIS-RESEARCH/Nepali-Text-Corpus"
SPLIT = "train"
TEXT_COLUMN = "Article"

print(f"Loading {DATASET_NAME} (split: {SPLIT})...")
ds = load_dataset(DATASET_NAME)
print("Columns available:", ds[SPLIT].column_names)
print(f"Total records: {len(ds[SPLIT]):,}")

## 4. Harvest Unigram Frequencies

In [ ]:
unigram_counts = Counter()

print("Starting unigram extraction...")
# Iterate over documents and update counter
for article in tqdm(ds[SPLIT][TEXT_COLUMN], desc="Processing articles"):
    tokens = tokenize_nepali_unigrams(article)
    unigram_counts.update(tokens)

print("Extraction complete.")
print(f"Total unique unigrams: {len(unigram_counts):,}")
print(f"Total token occurrences: {sum(unigram_counts.values()):,}")

print("\nTop 20 most frequent unigrams:")
for rank, (word, count) in enumerate(unigram_counts.most_common(20), 1):
    print(f"{rank:2d}. {word:<15} {count:,}")

## 5. Save Unigrams to File

Save unigram counts sorted in descending order to a tab-separated file (`unigram\tcount`).

In [ ]:
output_file = Path("unigrams.txt")
min_freq = 1  # Set threshold to filter out rare hapax legomena if needed

print(f"Saving unigrams to {output_file.resolve()}...")
with open(output_file, "w", encoding="utf-8") as f:
    for unigram, count in unigram_counts.most_common():
        if count >= min_freq:
            f.write(f"{unigram}\t{count}\n")

print(f"Saved {len(unigram_counts):,} unigrams to {output_file.name}.")